# 阿克曼公式（Ackermann's Formula）極點安置：逐格教學 Notebook

**範例系統：剛體衛星的航向軸姿態控制**

本 Notebook 是 `full-Ackermann-formula-example.md` 的**教學執行版**，設計成「**一格理論 → 一格程式 → 一格拆解**」的節奏，請由上而下依序執行每一格（`Shift + Enter`）。

每個步驟都會回答四件事：

1. **用了哪個公式**
2. **公式在說什麼**（物理意義）
3. **每個變數是什麼**（符號表）
4. **對應到哪一行程式**（數學 ↔ 程式對照表）

搭配閱讀：`full-Ackermann-formula-example.md`（完整推導與手算）、`full-Ackermann-formula-example.m`（精簡可執行版）、`../CHP1/chp1.md`（本例受控體的建模來源）。

---


## 先看這三行：本篇從頭到尾都在繞它們轉

**狀態方程式**（物理世界怎麼走，衛星自己會做的事）：

$$\mathbf x(k+1) = A\,\mathbf x(k) + B\,u(k)$$

**輸出方程式**（感測器看得到什麼）：

$$y(k) = C\,\mathbf x(k) + D\,u(k)$$

**控制律**（你要寫進微控制器的那一行）：

$$u(k) = -K\,\mathbf x(k) + N\,r(k)$$

整份 Notebook 的工作就是把這三行的每個字母填滿：

| 步驟 | 在做什麼 | 填滿了誰 |
|---|---|---|
| 1–2 | 物理建模 + 離散化 | $A,\ B,\ C,\ D$ |
| 3–4 | 下訂單 + 阿克曼公式 | $K$ |
| 5 | 消除穩態誤差 | $N$ |
| 6 | 寫成程式跑起來 | 三行一起動 |

### ⚠️ 最容易混淆的一點：大寫 $K$ 和小寫 $k$ 完全不同

$$u(\underbrace{k}_{\text{小寫：第幾步}}) = -\underbrace{K}_{\text{大寫：增益矩陣}}\,\mathbf x(k) + N\,r(k)$$

| 符號 | 是什麼 | 形狀 | 會不會變 | 本例 |
|---|---|---|---|---|
| **小寫 $k$** | **取樣序號**（第幾步），寫在括號裡 | 一個整數 | **每一步都變** | $1,2,3,\dots,50$ |
| **大寫 $K$** | **狀態回授增益矩陣**（權重） | $1\times2$ | **從頭到尾不變** | $[4\ \ 5.8]$ |

在程式裡它們是**兩個同時存在的變數**（Octave 大小寫敏感）——步驟 6 的迴圈跑起來時，左邊的 `k` 一直在跳，右邊的 `K` 一動也不動。**$K$ 是設計階段就算好、燒進韌體的常數；$k$ 是執行時的迴圈計數器。**

### 小寫 $k$ 到底是什麼？

- $k$ 是**取樣序號**：整數 $0,1,2,3,\dots$，代表「第幾次醒來」。
- $\mathbf x(k)$ 唸作「**第 $k$ 步的 $\mathbf x$**」。**那個括號是「編號」，不是乘法，也不是函數呼叫。**
- $\mathbf x(k+1)$ 是「**下一步的狀態**」，不是「$\mathbf x$ 加 1」。
- **實際時間** $t = k\times T$。本例 $T=0.1$ 秒，所以 $k=50$ 對應 $5$ 秒。
- 它就是程式裡 `for k = 1:steps` 的那個迴圈變數。

**為什麼用 $k$ 不用 $t$？** 連續時間寫 $\theta(t)$，$t$ 是實數，任何時刻都有定義。但數位系統只在 $t=0,\ T,\ 2T,\dots$ 這些孤立時刻存在，中間是空白的——用整數編號比較自然。嚴格說 $\mathbf x(k)$ 是 $\mathbf x(kT)$ 的簡寫。

### 那 $u(k)$ 的括號呢？

一模一樣：**第 $k$ 步的輸入**。$u(3)$ 是「第 3 步推力器該出多少轉矩」，不是「$u$ 乘以 3」。$y(k)$、$r(k)$ 也是同一個規則——**這三行式子裡所有的 $(k)$ 都是「第幾步」的標籤**。


## 🔧 環境設定

在 Octave Jupyter Notebook 中執行本範例需要：

1. `%plot --format svg` — Octave kernel 的 magic 語法（**必須放在 cell 第一行**），讓圖表內嵌顯示；SVG 對中文字型支援較好。
2. `pkg load control` — 載入 control 套件，提供 `ss`、`c2d`、`ssdata`、`acker`、`dcgain`、`ctrb` 等函數。
3. 設定中文字型，避免圖表標題亂碼。

> 若你在**標準 MATLAB**（非 Octave）中執行，`pkg load control` 這行會報錯，直接刪除即可——MATLAB 內建 Control System Toolbox。


In [ ]:
%plot --format svg

warning('off', 'Octave:gnuplot-graphics');   % 關閉繪圖後端的提醒訊息
warning('off', 'Octave:fltk-graphics');
graphics_toolkit('gnuplot');
clear; clc;
pkg load control;   % 若在 MATLAB 中執行，請刪除或註解此行

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

disp('環境就緒');

## 📖 全篇符號總表

先看這張表，後面遇到任何符號都可以回頭對照。

| 符號 | 程式變數 | **型別（形狀）** | 意義 | 單位 | 本例數值 |
|---|---|---|---|---|---|
| $J$ | `J` | **純量** $1\times1$ | 轉動慣量（抵抗旋轉的能力） | kg·m² | 2 |
| $T$ | `T` | **純量** $1\times1$ | **取樣週期**（電腦多久算一次） | s | 0.1 |
| $k$ | `k` | **純量**（整數） | 取樣序號，實際時間 $t=kT$ | — | 1…50 |
| $\theta(t)$ | — | **純量** $1\times1$ | 衛星航向角 | 度 | — |
| $u(k)$ | `u` | **純量** $1\times1$ ⚠️ | 推力器轉矩，系統**輸入**（一般為 $m\times1$ 向量，本例只有一個推力器故退化成純量） | N·m | 120 → −0.7 |
| $\mathbf x(k)$ | `x` | **行向量** $2\times1$ | **狀態向量**（角度與角速度打包在一起） | — | $[0.3;\ 6.0]$ |
| $x_1,\ x_2$ | `x(1), x(2)` | **純量** $1\times1$（各） | 狀態：角度、角速度 | 度, 度/s | — |
| $y(k)$ | `y` | **純量** $1\times1$ | 感測器讀值（一般為 $p\times1$ 向量） | 度 | — |
| $r$ | `r` | **純量** $1\times1$ | 目標角度指令 | 度 | 30 |
| $A_c$ | `Ac` | **矩陣** $2\times2$ | **連續時間**系統矩陣 | — | 見步驟 1 |
| $B_c$ | `Bc` | **行向量** $2\times1$ | **連續時間**輸入矩陣 | — | $[0;\ 0.5]$ |
| $C_c$ | `Cc` | **列向量** $1\times2$ | 輸出矩陣 | — | $[1\ \ 0]$ |
| $D_c$ | `Dc` | **純量** $1\times1$ | 直接傳遞矩陣 | — | 0 |
| $A$ | `A` | **矩陣** $2\times2$ | **離散時間**系統矩陣 | — | 見步驟 2 |
| $B$ | `B` | **行向量** $2\times1$ | **離散時間**輸入矩陣 | — | 見步驟 2 |
| $P$ | `P` | **列向量** $1\times2$ | **期望極點**（設計者的訂單） | — | $[0.8\ \ 0.9]$ |
| $\alpha_d(z)$ | `poly(P)` | **多項式**（係數 $1\times3$） | 期望特徵多項式 | — | $z^2-1.7z+0.72$ |
| $\mathcal{C}$ | `ctrb(A,B)` | **矩陣** $2\times2$ | 可控性矩陣（注意：與輸出矩陣 $C$ 不同東西） | — | 見步驟 3.5 |
| $K$ | `K` | **列向量** $1\times2$ ⚠️ | **狀態回授增益**（要求的答案；一般為 $m\times n$） | — | $[4\ \ 5.8]$ |
| $N$ | `N` | **純量** $1\times1$ | 參考輸入增益（消除穩態誤差） | — | 4 |

**形狀的通則**（$n$ = 狀態數、$m$ = 輸入數、$p$ = 輸出數；本例 $n=2,\ m=1,\ p=1$）：

| 符號 | 一般形狀 | 本例 | 誰決定它的形狀 |
|---|---|---|---|
| $\mathbf x$ | $n\times1$ | $2\times1$ | 狀態數 |
| $\mathbf u$ | $m\times1$ | $1\times1$（純量） | **致動器數量** |
| $\mathbf y$ | $p\times1$ | $1\times1$（純量） | **感測器數量** |
| $A$ | $n\times n$ | $2\times2$ | 狀態數 |
| $B$ | $n\times m$ | $2\times1$ | 狀態數 × 致動器數 |
| $C$ | $p\times n$ | $1\times2$ | 感測器數 × 狀態數 |
| $D$ | $p\times m$ | $1\times1$ | 感測器數 × 致動器數 |
| $K$ | $m\times n$ | $1\times2$ | **致動器數 × 狀態數** |

> **⚠️ 兩個最容易搞混的地方**
> - **大寫 $K$（增益矩陣，$1\times2$，恆定）vs 小寫 $k$（取樣序號，整數，每步遞增）**——Octave 大小寫敏感，程式裡是兩個變數。
> - **$K$ 有兩個元素，但 $u$ 只有一個數字**：$K\mathbf x$ 是 $(1\times2)(2\times1)=1\times1$，兩個權重**相加**成一個指令。$K$ 的元素個數由**狀態數**決定，$u$ 的元素個數由**致動器數**決定，兩者無關。

**會用到的 Octave 語法小抄**：

| 語法 | 意義 |
|---|---|
| `;` 在行尾 | 不要把結果印出來 |
| `;` 在 `[ ]` 內 | **換到矩陣的下一列**（同一個符號，用途完全不同） |
| `[0 1; 0 0]` | 2×2 矩陣，第一列 `0 1`、第二列 `0 0` |
| `[0; 1/J]` | 2×1 **行向量** |
| `A^2` | 矩陣的**平方**（$A\times A$），不是逐元素平方（逐元素是 `A.^2`） |
| `eye(2)` | 2×2 單位矩陣 $I$ |
| `inv(M)`, `det(M)`, `rank(M)` | 反矩陣、行列式、秩 |
| `x'` | 轉置 |

---


## 零、先建立三個直覺（新手必讀）

碰公式之前先講白三件事，想通了後面的數學只是把它們寫成矩陣。

### 直覺 1：「極點」決定系統的個性

任何線性系統的行為都由**分母的根（極點）**決定，極點就是系統的 DNA。

- **連續時間（$s$ 平面）**：極點 $s=-2$ → 響應含 $e^{-2t}$，會衰減。**實部 < 0 才穩定。**
- **離散時間（$z$ 平面）**：極點 $z=0.8$ → 響應含 $0.8^k$，每步乘 0.8 越來越小。**絕對值 < 1 才穩定。**

CHP1 的衛星 $G_p(s)=1/(Js^2)$ 兩個極點都在 $s=0$（離散後在 $z=1$），卡在穩定邊界 → 階躍響應**拋物線發散**，推一下永遠停不下來。

**本篇要做的事一句話：把這兩個爛極點，用回授硬生生搬到我們要的位置。** 這叫**極點安置**。

### 直覺 2：為什麼要離散化

物理世界連續（衛星每一瞬間都在轉），但微控制器每 $T=0.1$ 秒才醒來一次：讀感測器 → 算一次 → 輸出電壓 → 睡覺。

所以設計時**不能用連續模型 $A_c,B_c$，要用「站在微控制器視角的模型」$A,B$**：「這一步狀態是 $x(k)$，我輸出 $u(k)$ 並維持 0.1 秒，下一步會變成多少？」

### 直覺 3：$K$ 到底是什麼

$K$ 是一組**權重**。$u=-Kx$ 展開就是：

$$u = -\big(K_1\cdot\underbrace{\text{角度誤差}}_{x_1} \;+\; K_2\cdot\underbrace{\text{角速度}}_{x_2}\big)$$

- $K_1$：看到偏差就推回去 → **彈簧**，決定快慢。
- $K_2$：看到在動就踩煞車 → **阻尼**，決定震不震盪。

阿克曼公式做的事，就是把「我想要極點在 0.8 和 0.9」**直接反解成 $K_1,K_2$ 該是多少**，不用試誤。

---


## 一、步驟 1：連續時間狀態空間模型

### 用了哪個公式

**牛頓第二運動定律（旋轉形式）**：

$$J\,\ddot{\theta}(t) = v(t)$$

**狀態空間標準式**：

$$\dot{\mathbf{x}}(t) = A_c\mathbf{x}(t) + B_c v(t), \qquad y(t) = C_c\mathbf{x}(t) + D_c v(t)$$

### 公式在說什麼

第一式就是「轉矩 = 轉動慣量 × 角加速度」。太空中**沒有空氣阻力，沒有任何摩擦項**——這是整個問題困難的根源：系統自己不會停。

第二式把「一個二階微分方程」改寫成「兩個一階微分方程」。為什麼？因為電腦只會做矩陣乘法，不會解微分方程。寫成一階向量形式後，後面所有操作（離散化、極點安置、模擬）都只是矩陣運算。

### 怎麼推導

定義狀態變數 $x_1=\theta$（角度）、$x_2=\dot\theta$（角速度），則：

$$\dot{x}_1 = x_2 \quad\text{（角度變化率就是角速度——這是定義，不是物理）}$$

$$\dot{x}_2 = \frac{1}{J}v(t) \quad\text{（這一條才是牛頓定律）}$$

$$\begin{bmatrix}\dot{x}_1\\ \dot{x}_2\end{bmatrix}
=\underbrace{\begin{bmatrix}0&1\\0&0\end{bmatrix}}_{A_c}\begin{bmatrix}x_1\\x_2\end{bmatrix}
+\underbrace{\begin{bmatrix}0\\ 1/J\end{bmatrix}}_{B_c}v,
\qquad y=\underbrace{\begin{bmatrix}1&0\end{bmatrix}}_{C_c}\mathbf{x}$$


In [ ]:
%% 步驟 1：建立連續時間狀態空間模型
J  = 2;             % 轉動慣量 (kg-m^2)
Ac = [0 1; 0 0];    % 連續系統矩陣 A_c
Bc = [0; 1/J];      % 連續輸入矩陣 B_c
Cc = [1 0];         % 連續輸出矩陣 C_c（感測器只量得到角度 x1）
Dc = 0;             % 直接傳遞矩陣 D_c

sys_c = ss(Ac, Bc, Cc, Dc)

### 每個元素的意義（逐格解讀）

| 位置 | 值 | 意義 |
|---|---|---|
| $A_c(1,1)$ | 0 | 角度不會自己影響自己（沒有回正彈簧） |
| $A_c(1,2)$ | 1 | **角速度會累積成角度**——純運動學關係，恆為 1 |
| $A_c(2,1)$ | 0 | 角度不會產生轉矩（太空中沒有回正力） |
| $A_c(2,2)$ | 0 | **關鍵**：角速度不會自己衰減 → **沒有阻尼**。地面上的馬達這格會是負值（見 CHP1 第五節） |
| $B_c(1)$ | 0 | 轉矩無法「瞬間」改變角度（要先變成角速度） |
| $B_c(2)$ | $1/J=0.5$ | 轉矩產生角加速度，$J$ 越大加速越慢 |
| $C_c$ | $[1\ 0]$ | **只裝了角度計，量不到角速度** |
| $D_c$ | 0 | 沒有直通路徑；絕大多數物理系統 $D=0$ |

**數學 ↔ 程式對照**：

| 數學 | 程式 | 說明 |
|---|---|---|
| $\begin{bmatrix}0&1\cr0&0\end{bmatrix}$ | `[0 1; 0 0]` | `;` 在中括號內是**換列** |
| $\begin{bmatrix}0\cr1/J\end{bmatrix}$ | `[0; 1/J]` | 一個 `;` → 2×1 行向量 |
| $\dot x=A_cx+B_cu,\ y=C_cx+D_cu$ | `ss(Ac,Bc,Cc,Dc)` | 引數順序**永遠**是 A, B, C, D |

> **💡 常見混淆**：$C_c=[1\ 0]$ 是「**感測器**量得到什麼」，$K=[4\ 5.8]$ 是「**控制器**怎麼加權」，兩件完全不同的事，只是形狀剛好都是 1×2。看到 `C` 想「量測」，看到 `K` 想「控制」。

---


## 📦 深入理解 `ss()`：為什麼只有四個矩陣？

### 問題 1：不是兩條方程式嗎？為什麼只呼叫一次 `ss()`？

因為**那四個引數，就是那兩條方程式**。一條方程式各出兩個矩陣，剛好四個：

$$\underbrace{\dot{\mathbf x} = \boxed{A_c}\mathbf x + \boxed{B_c}u}_{\text{第 1 條：狀態方程}}
\qquad
\underbrace{y = \boxed{C_c}\mathbf x + \boxed{D_c}u}_{\text{第 2 條：輸出方程}}$$

```text
ss( Ac , Bc , Cc , Dc )
     └───┬───┘  └───┬───┘
      第 1 條方程   第 2 條方程
```

**為什麼是兩條？** 因為它們回答兩個不同的問題：

| 方程式 | 回答的問題 | 誰在乎 |
|---|---|---|
| $\dot{\mathbf x}=A_c\mathbf x+B_cu$ | **系統內部**怎麼演變 | 物理世界（衛星自己） |
| $y=C_c\mathbf x+D_cu$ | 我**能看到**其中哪些 | 感測器（角度計） |

**內部有什麼，跟你量得到什麼，是兩回事**——這也是後面需要**觀測器**的原因（$x_2$ 出現在第一條裡，卻不出現在第二條裡）。

### 問題 2：為什麼「剛好」是四個矩陣？

**順序是反過來的：**

```text
狀態空間模型在數學上被「定義」成這個固定樣板
        ↓
我把衛星「套」進模板（令 x1 = θ, x2 = θ̇）
        ↓
套完必定產生四個矩陣
```

$\dot{\mathbf x}=A\mathbf x+B\mathbf u,\ \mathbf y=C\mathbf x+D\mathbf u$ 這兩行是**定義**，不是任何特定系統的推導結果。`ss()` 只有四個欄位，純粹因為定義只有四個矩陣符號。

定義為什麼只有四項？因為 $4=2\text{ 條方程式}\times2\text{ 項}$，而**每條只能有兩項**——系統裡總共只有兩種東西（狀態 $\mathbf x$、輸入 $\mathbf u$）。不可能有第三項：第三項要乘什麼？$x^2$？$x\cdot u$？**那就不是線性系統了**（CHP1 的 SMIB 含 $\sin\phi$，所以必須先線性化）。

**維度也不是規定，是被矩陣運算逼出來的**（$n$=狀態數、$m$=輸入數、$p$=輸出數）：

| 要求 | 推得 |
|---|---|
| $A\mathbf x$ 必須和 $\dot{\mathbf x}$ 同為 $n\times1$ | $A$ 只能是 $n\times n$ |
| $B\mathbf u$ 必須是 $n\times1$，$\mathbf u$ 是 $m\times1$ | $B$ 只能是 $n\times m$ |
| $C\mathbf x$ 必須和 $\mathbf y$ 同為 $p\times1$ | $C$ 只能是 $p\times n$ |
| $D\mathbf u$ 必須是 $p\times1$ | $D$ 只能是 $p\times m$ |

本例 $n=2,m=1,p=1$ → $A$ 是 2×2、$B$ 是 2×1、$C$ 是 1×2、$D$ 是 1×1。這也解釋了 `a(2x2) and b(3x1) are incompatible`：不是 `ss` 有特別規矩，是**矩陣加法要求兩邊形狀一樣**。

**最反直覺的一點：系統變複雜，矩陣「變大」不「變多」**（下一格會實測）：

| 系統變化 | 誰變了 |
|---|---|
| 階數變高 | $A$ 從 2×2 → 10×10 |
| 多裝推力器 | $B$ 多幾**行** |
| 多裝感測器 | $C$ 多幾**列** |
| **矩陣個數** | **永遠是 4** |

> **例外**：廣義系統 $E\dot{\mathbf x}=A\mathbf x+B\mathbf u$ 多一個 $E$；時變系統 $A(t)$ 不是 LTI，裝不進去；非線性系統要先線性化。

### 問題 3：可以寫成 `ss(C, D, A, B)` 嗎？

**不能。`ss()` 靠「位置」認人，根本不知道你傳的矩陣是什麼。** 特別注意 `ss(Ac, Cc', Bc', Dc)`（B 與 C 對調）**完全不報錯**，卻建出一個完全不同的系統——就像 `rgb(255,0,0)`，函數不會「看出」你要紅色，是約定好第一個位置是紅色。

在 Octave `control-4.2.1`，**A、B、C、D 只能靠位置傳**；能用名字傳的只有附註資訊（`'stname'`、`'inname'`、`'outname'`）。取樣時間也必須用第 5 個位置。

**維度檢查抓得到「打錯字」，抓不到「觀念錯誤」。**

**為什麼 C、D 可以省略，A、B 不行？** 因為 $C=I$（全部狀態都量得到）、$D=0$（沒有直通路徑）是合理預設，而 $A$、$B$ **就是系統本身**，沒有預設值可言。

### 問題 4：什麼時候該呼叫 `ss()`？

> **只有當下一個要呼叫的函數「收系統物件」而不收散裝矩陣時，才需要打包。**

| 不需要 `ss()` | 一定要 `ss()` 物件 |
|---|---|
| `acker(A,B,P)` ✅ | `c2d(sys,T)` ❌ |
| `place(A,B,P)` ✅ | `dcgain(sys)` ❌ |
| `ctrb(A,B)` ✅ | `step(sys)` ❌ |
| `eig(A-B*K)` ✅ | `feedback`、`pole`、`bode` ❌ |

錯誤訊息 `'c2d' is a method of class 'lti'` 講明了原因：它不是一般函數，是 LTI 物件的**方法**。

**為什麼分兩堆？** 因為 A、B、C、D 其實**不足以**描述系統，還少一項：**這是連續還是離散？**

```matlab
A = [1 0.1; 0 1]     % 光看數字，分不出連續還是離散
```

`acker` 不在乎（純線性代數）；`c2d`、`dcgain` 非常在乎（連續代 $s=0$、離散代 $z=1$）。`ss()` 補的就是這個資訊：

| 寫法 | `tsam` | 判定 |
|---|---|---|
| `ss(A,B,C,D)` | 0 | **連續** |
| `ss(A,B,C,D,0)` | 0 | **連續**（明寫 0 等同不寫） |
| `ss(A,B,C,D,0.1)` | 0.1 | **離散**，每 0.1 秒一步 |
| `ss(A,B,C,D,-1)` | −1 | **離散**，取樣時間未指定 |

這就是步驟 5 的 `ss(A-B*K, B, C, D, T)` 那個 `T` 不能漏的原因——漏了 `tsam` 變 0，`dcgain` 會用 $s=0$ 而非 $z=1$，$N$ 直接算錯。

### 回到本程式：`ss()` 其實只出現兩次

```matlab
sys_c = ss(Ac, Bc, Cc, Dc);        % 打包，只因為下一行的 c2d 要物件
sys_d = c2d(sys_c, T, 'zoh');      % 用掉
[A, B, C, D] = ssdata(sys_d);      % 用完立刻「拆包」，變回散裝矩陣
K = acker(A, B, P);                % 散裝矩陣，不需要 ss
sys_cl = ss(A-B*K, B, C, D, T);    % 再打包，只因為 dcgain 要物件
N = 1 / dcgain(sys_cl);            % 用掉
for k = 1:steps
    u = -K * x + N * r;            % 全程散裝矩陣，完全沒碰 ss
    x = A * x + B * u;
end
```

模式：**打包 → 用掉 → 拆包 → 繼續用散裝**。`ssdata()` 就是 `ss()` 的反動作。

**燒進微控制器的那部分（`for` 迴圈）從頭到尾沒有 `ss`**——MCU 裡沒有 Octave，只有矩陣乘法。`ss` 純粹是設計階段給函式庫看的包裝紙。


In [ ]:
%% 動手實測：驗證上面四個問題的說法（可以自己改來玩）
disp('=== 實驗1：系統變複雜，矩陣會變多嗎？ ===');
n=10; m=3; p=2;                              % 10階、3輸入、2輸出
s10 = ss(rand(n,n), rand(n,m), rand(p,n), zeros(p,m));
printf('  A 是 %dx%d   B 是 %dx%d   C 是 %dx%d   D 是 %dx%d\n', ...
       rows(s10.a),columns(s10.a), rows(s10.b),columns(s10.b), ...
       rows(s10.c),columns(s10.c), rows(s10.d),columns(s10.d));
disp('  => 矩陣「個數」仍是 4 個，只有「尺寸」變大');

disp(' '); disp('=== 實驗2：位置放錯，會被抓到嗎？ ===');
try
  bad = ss(Ac, Cc', Bc', Dc);                % 故意把 B 跟 C 對調
  printf('  ss(Ac, Cc'', Bc'', Dc)  ->  沒有報錯！但 C 變成 [%g %g]，是完全不同的系統\n', ...
         bad.c(1), bad.c(2));
catch e
  printf('  X %s\n', strrep(e.message, "\n", " "));
end
try
  bad2 = ss(Cc, Dc, Ac, Bc);                 % 整個順序打亂
  disp('  ss(Cc,Dc,Ac,Bc) 成功');
catch e
  printf('  ss(Cc,Dc,Ac,Bc)      ->  X  %s\n', strrep(e.message, "\n", " "));
end
disp('  => 能不能抓到純粹看運氣：維度檢查抓得到打錯字，抓不到觀念錯誤');

disp(' '); disp('=== 實驗3：省略 C、D 會怎樣？ ===');
s2 = ss(Ac, Bc);
printf('  ss(Ac,Bc)  ->  C 自動補成 %s，D 自動補成 %s\n', mat2str(s2.c), mat2str(s2.d));

disp(' '); disp('=== 實驗4：第 5 引數決定連續還是離散 ===');
printf('  ss(...)      tsam=%-5g  離散?%d\n', ss(Ac,Bc,Cc,Dc).tsam,     isdt(ss(Ac,Bc,Cc,Dc)));
printf('  ss(...,0.1)  tsam=%-5g  離散?%d\n', ss(Ac,Bc,Cc,Dc,0.1).tsam, isdt(ss(Ac,Bc,Cc,Dc,0.1)));
printf('  ss(...,-1)   tsam=%-5g  離散?%d\n', ss(Ac,Bc,Cc,Dc,-1).tsam,  isdt(ss(Ac,Bc,Cc,Dc,-1)));

disp(' '); disp('=== 實驗5：哪些函數不需要 ss 物件？ ===');
Ad=[1 0.1;0 1]; Bd=[0.0025;0.05];
printf('  acker(A,B,P) 吃散裝矩陣  -> K = [%g %g]  OK\n', acker(Ad,Bd,[0.8 0.9]));
try
  c2d(Ac, Bc, 0.1);
catch e
  printf('  c2d(Ac,Bc,0.1) 吃散裝矩陣 -> X  %s\n', strtrim(strsplit(e.message, "\n"){1}));
end

## 二、步驟 2：ZOH 離散化

### 用了哪個公式

$$\boxed{A = e^{A_cT}, \qquad B = \left(\int_0^{T} e^{A_c\tau}d\tau\right)B_c}$$

$$\mathbf{x}(k+1) = A\mathbf{x}(k) + Bu(k)$$

### 公式在說什麼

$e^{A_cT}$ 是**矩陣指數**，定義為無窮級數：

$$e^{A_cT} = I + A_cT + \frac{(A_cT)^2}{2!} + \cdots$$

物理意義：「**不施加任何輸入時，經過 $T$ 秒狀態會自己演化成什麼樣**」。而 $B$ 的積分項是：「**這 $T$ 秒內，維持不變的輸入累積貢獻了多少**」。

**ZOH（Zero-Order Hold，零階保持器）**：微控制器每 0.1 秒才輸出一個數值，D/A 會把電壓**夾持住不變**直到下一個新值——所以輸入是階梯狀的。積分式能把 $u$ 提到外面，正是基於這個假設。（就是 CHP1 第二節飛機降落例子裡的 data hold。）

### 公式從哪來？（純量 → 矩陣，一步一步推）

這兩個公式不是憑空掉下來的，它就是**一階微分方程的標準解，換成矩陣寫法**。

**第一步：先看純量版本。** $\dot{x} = ax + bu$ 的標準解分兩部分：

- **齊次解**（令 $u=0$，系統自己演化）：$x(t)=e^{at}x(0)$
- **完整解**（含輸入，變數變異法）：

$$x(t)=\underbrace{e^{at}x(0)}_{\text{自己演化}}+\underbrace{\int_0^t e^{a(t-\tau)}\,b\,u(\tau)\,d\tau}_{\text{輸入的累積貢獻}}$$

第二項的意思是：過去每個時刻 $\tau$ 注入的 $bu(\tau)$，都要再乘上「從 $\tau$ 演化到 $t$」的因子 $e^{a(t-\tau)}$，然後全部加總。

**第二步：只看一個取樣區間**（$t$ 從 $kT$ 到 $kT+T$）。**這時 ZOH 假設發揮作用**——這 $T$ 秒內 $u(\tau)=u(k)$ 是常數，可以提到積分外面：

$$x(kT+T)=e^{aT}x(kT)+\left[\int_0^{T}e^{a(T-\lambda)}d\lambda\right]b\,u(k)$$

**第三步：換元** $\sigma=T-\lambda$，式子變乾淨：

$$x(k+1)=\underbrace{e^{aT}}_{\to\,A}x(k)+\underbrace{\left[\int_0^{T}e^{a\sigma}d\sigma\right]b}_{\to\,B}\,u(k)$$

**第四步：純量換矩陣。** $a\to A_c$、$b\to B_c$，$e^{at}$ 換成矩陣指數 $e^{A_ct}$（同一個級數定義），立刻得到 $A=e^{A_cT}$、$B=\left(\int_0^Te^{A_c\tau}d\tau\right)B_c$。

**這就是那兩個公式的全部來歷。** 矩陣指數不是憑空發明的新符號，它就是純量 $e^{at}$ 的矩陣推廣；「$B$ 為什麼要積分」也不神秘——純量版本本來就要積分。

> **💡 為什麼捷徑在本例不能用**：純量積分可直接算出 $\int_0^Te^{a\sigma}d\sigma=\frac{e^{aT}-1}{a}$，矩陣版對應 $B=A_c^{-1}(e^{A_cT}-I)B_c$，但這**要求 $A_c$ 可逆**。下一格會實測 CHP1 三個系統：衛星與馬達的 $\det A_c=0$（不可用），SMIB 的 $\det A_c=20$（可用）。
>
> **規律**：只要系統含**純積分器**（位置由速度積分而來、沒有回正彈簧），$A_c$ 第一行全是零 → 必定奇異。這正是本例要利用「$A_c$ 是冪零矩陣」手算積分的原因。

### 手算（本例可完全手算）

因為 $A_c^2=\mathbf{0}$（**冪零矩陣**），級數在第二項就截斷：

$$A = I + A_cT = \begin{bmatrix}1&T\\0&1\end{bmatrix} = \begin{bmatrix}1&0.1\\0&1\end{bmatrix}$$

$$\int_0^T e^{A_c\tau}d\tau = \int_0^T\begin{bmatrix}1&\tau\\0&1\end{bmatrix}d\tau = \begin{bmatrix}T&T^2/2\\0&T\end{bmatrix}$$

$$B = \begin{bmatrix}T&T^2/2\\0&T\end{bmatrix}\begin{bmatrix}0\\1/J\end{bmatrix}
= \begin{bmatrix}T^2/(2J)\\ T/J\end{bmatrix}
= \begin{bmatrix}0.0025\\0.05\end{bmatrix}$$

執行下一格，看程式算出來的是不是同一組數字。


In [ ]:
%% 步驟 2：ZOH 離散化
T = 0.1;                        % 取樣週期 (s)
sys_d = c2d(sys_c, T, 'zoh');   % continuous to discrete，零階保持器
[A, B, C, D] = ssdata(sys_d);   % 把四個矩陣從物件中取出

disp('離散系統矩陣 A ='); disp(A);
disp('離散輸入矩陣 B ='); disp(B);

% 與手算對照
printf('手算 A(1,2) = T      = %.4f\n', T);
printf('手算 B(1)   = T^2/2J = %.4f\n', T^2/(2*J));
printf('手算 B(2)   = T/J    = %.4f\n', T/J);

In [ ]:
%% 驗證推導：積分式 vs. 捷徑，以及為什麼本例不能用捷徑
disp('=== (a) 用數值積分驗證 B 的公式 ===');
Nn = 20000; h = T/Nn; S = zeros(2);
for i = 0:Nn
    w = 1; if (i == 0 || i == Nn), w = 0.5; end     % 梯形法權重
    S = S + w * expm(Ac*i*h) * h;                   % 累加 e^{Ac*tau} dtau
end
Bnum = S * Bc;
printf('  數值積分 (int_0^T e^{Ac tau} dtau)*Bc = [%.8f; %.8f]\n', Bnum(1), Bnum(2));
printf('  c2d 給的 B                            = [%.8f; %.8f]\n', B(1), B(2));
printf('  差異 = %.2e  => 推導出的公式與 c2d 完全一致\n', max(abs(Bnum - B)));

disp(' '); disp('=== (b) 捷徑 B = inv(Ac)*(e^{AcT}-I)*Bc 何時能用？ ===');
sysList = {[0 1; 0 0],      [0; 0.5], '衛星（純積分x2）    '; ...
           [0 1; 0 -2],     [0; 3],   '伺服馬達（積分+一階）'; ...
           [0 1; -20 -0.2], [0; 2],   'SMIB（彈簧阻尼）    '};
for i = 1:rows(sysList)
    Ai = sysList{i,1}; Bi = sysList{i,2}; nm = sysList{i,3};
    [~, Bd] = ssdata(c2d(ss(Ai, Bi, [1 0], 0), T, 'zoh'));
    printf('  %s det(Ac)=%7.2f  ', nm, det(Ai));
    if abs(det(Ai)) < 1e-12
        printf('奇異 -> 捷徑不可用，只能用積分式\n');
    else
        Bs = Ai \ (expm(Ai*T) - eye(2)) * Bi;
        printf('非奇異 -> 捷徑誤差 %.1e，可用\n', max(abs(Bd - Bs)));
    end
end
disp('  => 含純積分器的系統，Ac 第一行全零 -> 必定奇異');

### 這些數字的物理意義（這段最值得看）

把離散方程式的第一列攤開：

$$x_1(k+1) = \underbrace{1\cdot x_1(k)}_{\text{原本的角度}} + \underbrace{0.1\cdot x_2(k)}_{\text{角速度}\times T} + \underbrace{0.0025\cdot u(k)}_{\frac12 aT^2}$$

這就是**高中物理的等加速度公式** $s = s_0 + v_0t + \tfrac12at^2$！其中 $a=u/J$，所以 $\tfrac12aT^2 = \tfrac{T^2}{2J}u = 0.0025u$。

第二列：

$$x_2(k+1) = \underbrace{1\cdot x_2(k)}_{\text{角速度不會自己衰減}} + \underbrace{0.05\cdot u(k)}_{aT}$$

也就是 $v = v_0 + at$。**離散化矩陣不是玄學，它就是把牛頓運動學公式排進矩陣裡。**

| 元素 | 值 | 意義 |
|---|---|---|
| $A(1,1)$ | 1 | 角度完整保留（積分特性） |
| $A(1,2)$ | $T=0.1$ | 這 0.1 秒內角速度累積出多少角度 |
| $A(2,2)$ | 1 | **角速度完全不衰減** → 在 $z=1$ 有重根，臨界不穩定 |
| $B(1)$ | $T^2/2J=0.0025$ | 一步內轉矩貢獻的角度 |
| $B(2)$ | $T/J=0.05$ | 一步內轉矩貢獻的角速度 |

**數學 ↔ 程式對照**：

| 程式 | 意義 |
|---|---|
| `c2d(sys, T, 'zoh')` | 第三個引數是保持器型式；另有 `'foh'`、`'tustin'` |
| `ssdata(sys)` | 取回 A,B,C,D。**離散化只改變 A 和 B，C 與 D 不變**（量測關係與取樣無關） |

> **💡 取樣週期怎麼選？** $T$ 太大 → 兩次運算之間衛星已轉過頭，控制失效；$T$ 太小 → 算不完、雜訊被放大。慣例是**每個閉迴路時間常數內取樣 10～20 次**。本例最慢時間常數約 0.95 秒（見步驟 3），$T=0.1$ 剛好約 10 次，合理。

---


## 二之二、極點、特徵值、$s$、$z$ 到底什麼關係？

步驟 1、2 從頭到尾都在時域（微分方程、差分方程、矩陣），步驟 3 卻突然冒出 $s$ 平面、$z$ 平面。這一格補上中間的橋，先講結論：

> **極點安置從頭到尾都不需要離開時域。$s$ 和 $z$ 是為了「翻譯成人話」才引進的，不是計算必需品。**

### 0. 先問：$A-BK$ 是哪裡來的？

它**不是新概念，是代數化簡的結果**——把控制律代進狀態方程式、合併同類項：

```text
狀態方程式（物理給的）    x(k+1) = A·x(k) + B·u(k)
控制律（你寫的程式）      u(k)   = -K·x(k)
                          ↓ 代入
                          x(k+1) = A·x(k) + B·(-K·x(k))
                          ↓ 提出 x(k)
                          x(k+1) = (A - B·K)·x(k)
```

$$\mathbf x(k+1)=A\mathbf x(k)+B\underbrace{u(k)}_{=-K\mathbf x(k)}=(A-BK)\,\mathbf x(k)$$

**注意 $BK$ 是 $2\times2$ 矩陣，不是純量**（$2\times1$ 乘 $1\times2$ 是外積；反過來 $KB$ 才是 $1\times1$）。下一格會實測，並驗證「先算 u 再更新」與「直接乘 $A-BK$」走 100 步完全等價。

**精髓：回授「換掉」了系統矩陣。**

| | 系統矩陣 | 誰決定 |
|---|---|---|
| 開迴路 | $A$ | **硬體**——轉動慣量 $J$、取樣週期 $T$，你改不了 |
| 閉迴路 | $A-BK$ | **硬體 + 軟體**——$K$ 是你程式裡的兩個數字 |

$A$ 是衛星出廠就決定的，但接上回授後系統實際表現出來的矩陣變成 $A-BK$，而 $K$ 在你手上。這就是狀態回授的全部威力：**用軟體改造硬體的動態特性**。

**最漂亮的一點：看第二列**（角速度怎麼更新）：

```text
開迴路 x2(k+1) =  0.000*x1 + 1.000*x2     <- 沒有彈簧，角速度不衰減
閉迴路 x2(k+1) = -0.200*x1 + 0.710*x2     <- 出現彈簧與阻尼
```

$-0.2\,x_1$ 是**彈簧**（角度越偏，角速度越被往反方向拉），開迴路是 0，因為太空中沒東西把衛星拉回來；$0.71\,x_2$ 是**阻尼**（角速度每步只剩 71%），開迴路是 1，因為太空中沒有摩擦。

**回授用軟體，在真空中造出了原本不存在的彈簧和阻尼**——這正是直覺 3 說的 $K_1$ 像彈簧、$K_2$ 像阻尼。

加上 $N$ 之後的完整式子：

$$\mathbf x(k+1)=\underbrace{(A-BK)}_{\text{決定「怎麼動」}}\mathbf x(k)+\underbrace{BN\,r(k)}_{\text{決定「往哪去」}}$$

$N$ 只出現在**驅動項**裡，完全不碰 $A-BK$，所以 **$N$ 不影響極點**——這也是為什麼 `acker(A,B,P)` 只需要 $A$、$B$、$P$。

### 1. 「極點」就是「特徵值」

步驟 4 驗證設計成功用的是 `eig(A-B*K)`，**不是 z 轉換**。所謂「把極點安置到 0.8」，字面意思就是「讓 $A-BK$ 的特徵值等於 0.8」。純線性代數，跟頻域無關。

### 2. 特徵值 0.8 的字面意思：每一步乘 0.8

無輸入時 $\mathbf x(k)=A^k\mathbf x(0)$。沿著特徵向量方向，$A$ 作用一次就是乘一次 $\lambda$。所以 $|\lambda|<1$ 才穩定——**因為只有 $\lambda^k$ 才會縮小**，不是因為什麼「單位圓」的抽象規則。單位圓只是把這件事畫成圖。

### 3. 那 $z$ 是什麼？同一個東西的另一個名字

離散轉移函數 $G(z)=C(zI-A)^{-1}B+D$ 的**分母就是 $\det(zI-A)$**，它的根正是 $A$ 的特徵值。下一格會實測驗證：`eig(A-B*K)` 與 `pole(G(z))` 完全相同（差異 $10^{-16}$ 等級），而 $G(z)$ 的分母就是 $z^2-1.7z+0.72$。

**所以你給的 0.8、0.9 確實就是 z 轉移函數的極點——但同時也就是特徵值，不必先做 z 變換才能談它們。** 整份文件你都可以把「z 平面的極點」讀成「特徵值」。

> **小提醒**：嚴格說，若有零極點對消，轉移函數的極點會比特徵值少（消失的叫「不可控／不可觀」模態）。本例沒有這個問題。

### 4. $z=e^{sT}$ 的來歷——推導也在時域

連續系統解是 $e^{A_ct}$，特徵值 $s$ 對應 $e^{st}$；離散系統解是 $A^k$，特徵值 $z$ 對應 $z^k$。**同一個物理過程，在 $t=kT$ 必須給出同樣的數字**，所以 $z^k=e^{skT}\Rightarrow z=e^{sT}$。**沒有用到任何轉換**，就是兩條指數在取樣點上對齊（下一格會逐項印出來比對）。

### 5. 那為什麼還要引進 $z$ 平面？

1. **溝通**：教科書、同事都講「$z$ 平面極點」「單位圓內」，你要聽得懂。
2. **換算成秒**：$0.8$ 沒有單位、沒有感覺；換成 $\tau=0.448$ 秒才知道「大概半秒」。
3. **一眼判穩定**：開迴路 $1,1$ 在單位圓上 → 臨界不穩；閉迴路 $0.8,0.9$ 在圓內 → 穩定。**單位圓給了你一把尺。**

**這一步在設計上可以完全跳過**——直接寫 `P = [0.8, 0.9]` 也算得出一樣的 $K$。

### 6. $T$ 一改，$z$ 就跟著改

離散化時 $T$ 只進到 $A$、$B$（$C$、$D$ 是量測關係，與取樣無關）。**$A$ 內建了 $T$，所以特徵值也內建了 $T$**——這就是 $z=e^{sT}$ 裡有 $T$ 的原因。同樣的物理速度 $s$，取樣越快 $z$ 越靠近 1（下一格實測）。

所以「$z=0.8$ 快不快」這句話沒有意義，除非同時說明 $T$——這也是為什麼 $T$ 一改，$P$ 必須重算。

### 7. 誰是輸入、誰是輸出、誰是驗算

```text
   你給的（輸入）            算出來的（輸出）           驗算
┌───────────────┐     ┌───────────────┐     ┌───────────────┐
│  A, B  系統    │     │               │     │               │
│  P = [0.8 0.9] │ ──► │  K = [4 5.8]  │ ──► │  eig(A-B*K)   │
│    我的訂單     │acker│   完美權重     │     │  = 0.8, 0.9   │
└───────────────┘     └───────────────┘     └───────────────┘
                                              應該等於 P
```

`eig(A-B*K)` 的 0.8、0.9 **不是設定值，是量出來的**——量出來剛好等於訂單，才代表設計成功。下一格會換不同訂單、也會亂填 $K$，讓你看清楚誰牽動誰。

### 8. 所以整件事的邏輯是「反解」

| 方向 | 問題 | 名稱 | 工具 |
|---|---|---|---|
| 正向 | 給我 $K$，特徵值是多少？ | **分析** | `eig()` |
| 反向 | 給我特徵值，$K$ 該是多少？ | **設計** | **阿克曼公式** |

就像解方程式時「已知根，反求係數」。`eig` 則是把答案代回原式驗算——這也是為什麼步驟 4 的 `eig(A-B*K)` 標的是「**驗證**」而不是「計算」。


In [ ]:
%% 實測：A-BK 從哪來 / 特徵值 / z 平面 / 誰是輸入誰是輸出
Kv = acker(A, B, [0.8 0.9]); Acl = A - B*Kv;

disp('=== (0) A-BK 是哪裡來的？ ===');
printf('  B 是 %dx%d，K 是 %dx%d  ->  B*K 是 %dx%d（外積，不是純量）\n', ...
       rows(B),columns(B), rows(Kv),columns(Kv), rows(B*Kv),columns(B*Kv));
disp('  B*K ='); disp(B*Kv);
disp('  A   ='); disp(A);
disp('  A-B*K ='); disp(Acl);
x1 = [3; -1]; x2 = x1; dmax = 0;
for k = 1:100
    u = -Kv*x1;  x1 = A*x1 + B*u;        % 走法一：先算 u 再更新
    x2 = Acl*x2;                          % 走法二：直接乘 A-BK
    dmax = max(dmax, max(abs(x1 - x2)));
end
printf('  兩種走法跑 100 步，最大差異 = %g  => 完全等價\n', dmax);
printf('  第二列（角速度更新）：\n');
printf('    開迴路 x2(k+1) = %6.3f*x1 + %6.3f*x2   <- 沒有彈簧，角速度不衰減\n', A(2,1), A(2,2));
printf('    閉迴路 x2(k+1) = %6.3f*x1 + %6.3f*x2   <- 回授造出了彈簧與阻尼\n', Acl(2,1), Acl(2,2));
disp(' ');

disp('=== (1) 兩種說法，同一組數字 ===');
Gz = tf(ss(Acl, B, C, D, T));            % 轉成 z 轉移函數
printf('  時域： eig(A-B*K) = [%.4f  %.4f]\n', sort(eig(Acl)));
printf('  z域：  pole(G(z)) = [%.4f  %.4f]\n', sort(pole(Gz)));
printf('  差異 = %.2e  => 完全相同\n', max(abs(sort(eig(Acl)) - sort(pole(Gz)))));
[~, den] = tfdata(Gz, 'v');
printf('  G(z) 分母係數           = [%g %g %g]\n', den);
printf('  poly(eig(A-B*K))       = [%g %g %g]\n', poly(eig(Acl)));
printf('  (z-0.8)(z-0.9) 展開     = [%g %g %g]\n', poly([0.8 0.9]));

disp(' '); disp('=== (2) 特徵值 0.8 = 每一步乘 0.8 ===');
printf('  k    :'); for k = 0:5, printf('%9d', k); end
printf('\n  0.8^k:'); for k = 0:5, printf('%9.5f', 0.8^k); end
printf('\n');

disp(' '); disp('=== (3) z = e^{sT} 的來歷：兩條指數在取樣點上對齊 ===');
sc = log(0.8)/T;
printf('  s = ln(0.8)/T = %.4f rad/s\n', sc);
printf('  e^{s*kT}:'); for k = 0:4, printf('%10.5f', exp(sc*k*T)); end
printf('\n  0.8^k   :'); for k = 0:4, printf('%10.5f', 0.8^k); end
printf('\n');

disp(' '); disp('=== (4) T 一改，同樣的物理速度對到不同的 z ===');
for T2 = [0.2 0.1 0.05 0.01]
    printf('  T=%.2f s -> z = e^{sT} = %.5f\n', T2, exp(sc*T2));
end

disp(' '); disp('=== (5) 誰是輸入？換訂單，K 跟著變，eig 永遠追著新訂單 ===');
printf('  %-18s %-24s %s\n', '我給的 P (輸入)', 'acker 算的 K (輸出)', 'eig(A-B*K) (驗算)');
for Pc = {[0.8 0.9], [0.5 0.6], [0.95 0.99], [0.2 0.3]}
    p = Pc{1}; Kk = acker(A,B,p); e = sort(eig(A-B*Kk))';
    printf('  [%.2f %.2f]         [%8.3f %8.3f]        [%.4f %.4f]\n', p, Kk, e);
end

disp(' '); disp('=== (6) 不用 acker，亂填 K，eig 就落在別處 ===');
for Kx = {[4 5.8], [1 1], [0 0], [50 3]}
    kk = Kx{1}; e = sort(eig(A-B*kk))';
    printf('  K=[%5.1f %5.1f] -> eig = [%.4f %.4f]\n', kk, e);
end
disp('  => eig 是量出來的結果，不是設定值。K=[0 0] 就回到原本 z=1 的爛極點');

## 三、步驟 3：設定期望極點（向系統「下訂單」）

### 用了哪個公式

**s 平面 ↔ z 平面的翻譯字典**：

$$\boxed{z = e^{sT} \quad\Longleftrightarrow\quad s = \frac{\ln z}{T}}$$

**期望特徵多項式**：

$$\alpha_d(z) = (z-p_1)(z-p_2) = z^2 - (p_1+p_2)z + p_1p_2$$

### 公式在說什麼

**先接回上一格**：這裡要「訂」的極點，就是 $A-BK$ 的**特徵值**——時域的東西，不必先做 z 變換才能談。之所以還是用 $z$ 這個符號，是因為它同時也是 $G(z)$ 的極點，而且單位圓給了我們一把判斷快慢的尺。

第一式是連續與離散世界的翻譯。你在物理上想的是「我要幾秒內穩定」（$s$ 平面的語言），但下訂單時得換算成 $z$ 平面的數值。

第二式把「我要的根」還原成「我要的分母」。因為**極點安置的本質，就是強迫閉迴路的分母變成 $\alpha_d(z)$**。


In [ ]:
%% 步驟 3：設定期望極點
P = [0.8, 0.9];          % z 平面上的期望極點

alpha_d = poly(P);       % 由根反算多項式係數（高次→低次）
printf('期望特徵多項式係數 = [%g  %g  %g]  →  z^2 %+g z %+g\n', ...
       alpha_d(1), alpha_d(2), alpha_d(3), alpha_d(2), alpha_d(3));

% 翻譯回連續時間，看看這代表多快
s_poles = log(P) / T;
tau     = -1 ./ s_poles;
printf('\nz = %.1f  →  s = %8.4f rad/s  →  時間常數 tau = %.4f s\n', ...
       [P; s_poles; tau]);
printf('\n主極點（最慢的那個）時間常數 = %.4f s，4*tau = %.2f s 可安定到 2%%\n', ...
       max(tau), 4*max(tau));

### 結果解讀

$$s_1=\frac{\ln 0.8}{0.1}=-2.231 \Rightarrow \tau_1=0.448\,\text{s}, \qquad
s_2=\frac{\ln 0.9}{0.1}=-1.054 \Rightarrow \tau_2=0.949\,\text{s}\ (\textbf{主極點})$$

$0.9$ 衰減得比較慢（離單位圓比較近），所以它主宰整體反應速度，稱為**主極點**（dominant pole）。$4\tau\approx3.8$ 秒可安定到 2% 以內——後面模擬結果（5 秒到 29.67 度）會驗證這件事。

$$\alpha_d(z) = (z-0.8)(z-0.9) = z^2 - 1.7z + 0.72$$

### 極點該怎麼選？（新手最常卡在這裡）

| 極點位置 | 系統行為 | 代價 |
|---|---|---|
| 接近 1（如 0.99） | 非常慢，很溫和 | 太慢，可能不符規格 |
| 中等（0.8～0.9） | 適中 | **本例的選擇** |
| 接近 0（如 0.1） | 極快 | **控制訊號爆大**，推力器會飽和 |
| 負實數或 $| z|>1$ | 每步變號跳動／發散 | 不可用 |

**核心取捨**：極點越靠近原點越快，但所需推力越大。天下沒有白吃的午餐——這就是為什麼實務上不會盲目把極點推到 0。

**兩個實數極點 vs. 一對共軛複數極點**：本例選兩個**相異實數**極點，響應**不會震盪**。若規格是「阻尼比 $\zeta$、自然頻率 $\omega_n$」，就要先在 $s$ 平面算 $s=-\zeta\omega_n\pm j\omega_n\sqrt{1-\zeta^2}$，再用 $z=e^{sT}$ 換算成共軛複數 $z$ 極點。

---


## 四、步驟 3.5：可控性檢查（阿克曼公式的前提）

原始 `.m` 檔沒有這步，但**這是 `acker` 能成立的必要條件**，教科書必考、實務出錯也多半卡在這裡。

### 用了哪個公式

$$\mathcal{C} = \begin{bmatrix}B & AB & A^2B & \cdots & A^{n-1}B\end{bmatrix}, \qquad \text{可控} \iff \operatorname{rank}\mathcal{C}=n$$

### 公式在說什麼

**可控性**問的是：「只靠這一個推力器，我**有沒有能力**把系統從任意狀態搬到任意其他狀態？」

矩陣每一行的意義是「輸入的影響經過 0 步、1 步、2 步…傳播後長什麼樣」。若這些方向**張不滿整個狀態空間**（rank 不足），代表有些方向輸入永遠碰不到——那些方向的極點就**搬不動**，`acker` 會失敗（回傳警告或天文數字）。

### 手算

$$AB=\begin{bmatrix}1&0.1\\0&1\end{bmatrix}\begin{bmatrix}0.0025\\0.05\end{bmatrix}=\begin{bmatrix}0.0075\\0.05\end{bmatrix},\qquad
\mathcal{C}=\begin{bmatrix}0.0025&0.0075\\0.05&0.05\end{bmatrix}$$

$$\det\mathcal{C}=0.0025(0.05)-0.0075(0.05)=-0.00025\neq0$$

$$\mathcal{C}^{-1}=\frac{1}{-0.00025}\begin{bmatrix}0.05&-0.0075\\-0.05&0.0025\end{bmatrix}=\begin{bmatrix}-200&30\\200&-10\end{bmatrix}$$


In [ ]:
%% 步驟 3.5：可控性檢查
Cm = ctrb(A, B);        % 等同於 [B  A*B]
disp('可控性矩陣 Cm = [B  A*B] ='); disp(Cm);
printf('rank(Cm) = %d   (需要等於狀態數 n = %d)\n', rank(Cm), size(A,1));
printf('det(Cm)  = %g   (不可為 0)\n', det(Cm));

if rank(Cm) == size(A,1)
    disp('=> 系統完全可控，可以放心使用阿克曼公式');
else
    disp('=> 系統不可控，極點無法任意安置！');
end

disp(' '); disp('inv(Cm) ='); disp(inv(Cm));

### 對照

| 數學 | 程式 |
|---|---|
| $\mathcal{C}=[B\ \ AB]$ | `ctrb(A, B)` 或直接寫 `[B A*B]` |
| $\operatorname{rank}\mathcal{C}$ | `rank(Cm)` |
| $\det\mathcal{C}$ | `det(Cm)` |
| $\mathcal{C}^{-1}$ | `inv(Cm)` |

`rank = 2 = n` 且 `det ≠ 0` → **完全可控**。下一步的 $\mathcal{C}^{-1}$ 存在，阿克曼公式可以用。

---


## 五、步驟 4：阿克曼公式（本篇核心）

### 用了哪個公式

$$\boxed{K = \begin{bmatrix}0&0&\cdots&0&1\end{bmatrix}\;\mathcal{C}^{-1}\;\alpha_d(A)}$$

其中 $\alpha_d(A)$ 是把**期望特徵多項式的變數 $z$ 換成矩陣 $A$**：

$$\alpha_d(z)=z^n+a_{n-1}z^{n-1}+\cdots+a_0 \;\Longrightarrow\; \alpha_d(A)=A^n+a_{n-1}A^{n-1}+\cdots+a_0I$$

### 三個組件逐一解讀

| 組件 | 意義 |
|---|---|
| $[0\ \cdots\ 0\ 1]$ | 「取最後一列」的選擇向量。本例 $n=2$ → $[0\ \ 1]$ |
| $\mathcal{C}^{-1}$ | **把「數學上想要的變化」翻譯成「輸入該出多少力」**。這也是為什麼系統必須可控——不可控 → $\mathcal{C}$ 不可逆 → 公式直接爆掉 |
| $\alpha_d(A)$ | **整個公式的靈魂**。衡量「目前這顆真實的 $A$，離我想要的特徵多項式**差多遠**」。若 $A$ 的極點恰好已是我要的，由**凱萊–漢彌爾頓定理**可知 $\alpha_d(A)=\mathbf 0$，於是 $K=\mathbf 0$——系統已完美，不需要控制 |

**它憑什麼有效？** 目標是讓 $A-BK$ 的特徵多項式等於 $\alpha_d(z)$。推導方式是先把系統轉到**可控標準式**（此時 $K$ 可以用眼睛看出來，就是期望係數與原係數的差），再轉換回原座標。上面那行公式就是整套操作合併化簡後的結果。**你不必會推導，但要記得它做的事：一步反解出 $K$。**

### 手算（請跟著算一遍，最有感）

**第一步**：$\alpha_d(A)=A^2-1.7A+0.72I$，其中 $A^2=\begin{bmatrix}1&0.2\\0&1\end{bmatrix}$

$$\alpha_d(A)=\begin{bmatrix}1-1.7+0.72 & 0.2-0.17\\ 0 & 1-1.7+0.72\end{bmatrix}=\begin{bmatrix}0.02&0.03\\0&0.02\end{bmatrix}$$

**第二步**：取 $\mathcal{C}^{-1}$ 的最後一列：$[0\ \ 1]\begin{bmatrix}-200&30\\200&-10\end{bmatrix}=[200\ \ -10]$

**第三步**：

$$K=[200\ \ -10]\begin{bmatrix}0.02&0.03\\0&0.02\end{bmatrix}=[\,4\quad 5.8\,]$$

下一格用兩種方式算，看是否一致。


In [ ]:
%% 步驟 4：阿克曼公式
% --- 方法 A：內建函數，一行解決 ---
K = acker(A, B, P);
printf('K = acker(A,B,P)  =  [%g  %g]\n', K(1), K(2));

% --- 方法 B：照公式手動做一次，驗證原理 ---
alphaA   = A^2 - 1.7*A + 0.72*eye(2);   % alpha_d(A)，注意 A^2 是矩陣平方
disp(' '); disp('alpha_d(A) = A^2 - 1.7A + 0.72I ='); disp(alphaA);

e_n      = [0 1];                        % 選擇向量 [0 ... 0 1]
K_manual = e_n * inv(Cm) * alphaA;
printf('\nK 手動公式        =  [%g  %g]\n', K_manual(1), K_manual(2));
printf('兩者最大差異      =  %g   (應為 0 或極小的浮點誤差)\n', max(abs(K - K_manual)));

### 結果的物理意義

$$u(k) = -4\,x_1(k) - 5.8\,x_2(k)$$

- $K_1=4$：每偏差 1 度，施加 4 N·m 回正轉矩（**彈簧**）。
- $K_2=5.8$：每有 1 度/秒角速度，施加 5.8 N·m 反向煞車轉矩（**阻尼**）。

$K_2>K_1$ 說明這個設計**偏重煞車**——這正是無阻尼系統（太空中沒摩擦）必須付的代價：不主動煞車，衛星就會衝過頭無限震盪。

**數學 ↔ 程式對照**：

| 數學 | 程式 |
|---|---|
| $\alpha_d(A)=A^2-1.7A+0.72I$ | `A^2 - 1.7*A + 0.72*eye(2)` |
| $[0\ \ 1]$ | `[0 1]` |
| $K=[0\ 1]\mathcal{C}^{-1}\alpha_d(A)$ | `[0 1] * inv(Cm) * alphaA` |

> **⚠️ `acker` vs `place`**：`acker` 直接照公式做，含 $\mathcal{C}^{-1}$ 與 $A$ 的高次冪，當階數 $n$ 大於約 5～10 時**數值誤差嚴重惡化**（可控性矩陣通常病態）；而且 `acker` **只適用單輸入系統**，不允許重複極點。實務請改用 `place()`（數值穩定、支援多輸入）。本例 $n=2$ 用 `acker` 完全沒問題，而且它最能看清楚極點安置的原理。

---


### 驗證：極點真的被搬過去了嗎？

$$A-BK=\begin{bmatrix}1&0.1\\0&1\end{bmatrix}-\begin{bmatrix}0.0025\\0.05\end{bmatrix}[4\ \ 5.8]
=\begin{bmatrix}0.99&0.0855\\-0.2&0.71\end{bmatrix}$$

2×2 矩陣的快速檢查法（**跡 = 特徵值之和，行列式 = 特徵值之積**）：

$$\operatorname{tr}=0.99+0.71=1.70=0.8+0.9\  \qquad \det=0.7029+0.0171=0.72=0.8\times0.9\ $$


In [ ]:
%% 驗證：閉迴路極點
Acl = A - B*K;          % 注意 B*K 是 (2x1)(1x2) = 2x2 矩陣，不是純量
disp('閉迴路矩陣 A - B*K ='); disp(Acl);

printf('trace(A-BK) = %.4f   應等於 0.8+0.9 = %.4f\n', trace(Acl), sum(P));
printf('det(A-BK)   = %.4f   應等於 0.8*0.9 = %.4f\n', det(Acl), prod(P));
disp(' ');
disp('eig(A - B*K) ='); disp(eig(Acl));
disp('原本開迴路的極點 eig(A) ='); disp(eig(A));

### 解讀

開迴路 `eig(A) = [1, 1]`——兩個重根**卡在單位圓上**，這就是「推一下停不下來、階躍響應拋物線發散」的數學原因。

閉迴路 `eig(A-BK) = [0.9, 0.8]`——**極點確實被安置到訂單位置**，兩個都在單位圓內，系統穩定。

回授做的事就是這麼直接：**改寫系統的分母**。

---


## 六、步驟 5：參考輸入增益 $N$

### 為什麼還需要 $N$？

到目前為止的 $u=-Kx$ 只會把衛星**拉回 0 度**——因為把所有狀態逼到零，就是「回到原點」的意思。但我們要的是「轉到 30 度」。

最直覺的做法是 $u=-Kx+r$，但這樣**穩態值不會剛好是 30**。所以需要一個比例補償 $N$，讓閉迴路的**直流增益恰好為 1**（輸入 30 就穩在 30）。

### 用了哪個公式

閉迴路狀態方程：$\mathbf{x}(k+1)=(A-BK)\mathbf{x}(k)+BN\,r(k)$

穩態時 $\mathbf{x}(k+1)=\mathbf{x}(k)=\mathbf{x}_{ss}$，解出：

$$\mathbf{x}_{ss}=(I-A+BK)^{-1}BN\,r \;\Longrightarrow\; y_{ss}=C(I-A+BK)^{-1}BN\,r$$

要求 $y_{ss}=r$，因此：

$$\boxed{N=\frac{1}{C(I-A+BK)^{-1}B}=\frac{1}{\text{閉迴路直流增益}}}$$

（離散系統的直流增益就是把 $z=1$ 代進去，因為 $z=e^{sT}$ 在 $s=0$ 時等於 1。）

### 手算

$$I-(A-BK)=\begin{bmatrix}0.01&-0.0855\\0.2&0.29\end{bmatrix},\qquad \det=0.0029+0.0171=0.02$$

$$C(I-A+BK)^{-1}B=\frac{1}{0.02}[0.29\ \ 0.0855]\begin{bmatrix}0.0025\\0.05\end{bmatrix}=\frac{0.005}{0.02}=0.25
\;\Longrightarrow\; N=\frac{1}{0.25}=4$$


In [ ]:
%% 步驟 5：計算參考輸入增益 N
sys_cl  = ss(Acl, B, C, D, T);   % 第五個引數 T：宣告這是取樣週期 T 的離散系統
dc_gain = dcgain(sys_cl);        % 離散系統的 DC gain = z=1 時的增益
N       = 1 / dc_gain;

printf('閉迴路直流增益 = %.4f\n', dc_gain);
printf('N = 1/dcgain   = %.4f\n', N);

% 手算驗證
dc_manual = C * inv(eye(2) - Acl) * B + D;
printf('\n手算 C(I-A+BK)^-1 B = %.4f  (應與上面相同)\n', dc_manual);

% 有趣的觀察
printf('\nN = %g,  K(1) = %g   →  兩者相等！為什麼？見下一格\n', N, K(1));

### 為什麼 $N$ 剛好等於 $K_1$？（漂亮的物理洞察）

$N=4=K_1$ 不是巧合。**衛星是剛體，在太空中「維持」某個固定角度不需要任何轉矩**（沒有重力、沒有摩擦要對抗），所以穩態時必然 $u_{ss}=0$。而穩態狀態是 $\mathbf{x}_{ss}=[r,\ 0]^T$（角度到位、角速度歸零），代入控制律：

$$u_{ss}=-K_1r-K_2\cdot 0+Nr=(N-K_1)r=0 \;\Longrightarrow\; N=K_1$$

**這對任何「輸出即第一個狀態、且開迴路含純積分」的系統都成立**（衛星、無摩擦定位平台）。但對**有摩擦或有重力**的系統（CHP1 的伺服馬達、倒單擺）就**不成立**——那些系統維持位置需要持續施力，$u_{ss}\neq0$，必須老實用 `dcgain` 算 $N$。

**數學 ↔ 程式對照**：

| 數學 | 程式 | 說明 |
|---|---|---|
| $A-BK$ | `A - B*K` | `B*K` 是 (2×1)(1×2) = **2×2 矩陣**，不是純量 |
| 離散系統宣告 | `ss(Acl, B, C, D, T)` | 加上第五個引數 `T` 才是離散系統；不加預設為連續 |
| $C(I-A_{cl})^{-1}B+D$ | `dcgain(sys_cl)` | 對離散系統即 $z=1$；對連續系統則是 $s=0$ |

> **⚠️ $N$ 的弱點**：$N$ 是**前饋**（feedforward）補償，完全依賴模型準確。若真實的 $J$ 與你以為的不同，或存在恆定外力干擾，$N$ 就算錯了，穩態誤差會回來。要根本解決必須在控制器中加入**積分作用**——那是後續章節的主題。

---


## 七、步驟 6：控制律實作與模擬迴圈

### 用了哪兩個公式

**控制律**（微控制器每一步要算的）：

$$u(k)=-K\mathbf{x}(k)+N\,r(k)$$

**狀態更新方程式**（物理硬體的反應）：

$$\mathbf{x}(k+1)=A\mathbf{x}(k)+Bu(k), \qquad y(k)=C\mathbf{x}(k)$$

### 概念上最重要的一件事

這個 `for` 迴圈裡有**兩種身分**混在一起，一定要分清楚：

| 程式行 | 身分 | 真實系統中由誰執行 |
|---|---|---|
| `u = -K*x + N*r;` | **控制器（軟體）** | 微控制器裡真正跑的那幾行程式碼 |
| `x_next = A*x + B*u;` | **受控體（物理）** | 真實世界的衛星自己會做的事，**不是程式碼** |
| `y = C*x;` | **感測器** | 角度計 |

模擬中我們必須自己扮演物理世界（所以要寫 `x_next = A*x+B*u`），但**燒進微控制器的實際程式碼只有 `u = -K*x + N*r` 這一行**。這就是極點安置法在嵌入式系統上受歡迎的原因：設計階段的所有數學複雜度，最後濃縮成兩次乘法、一次加法。


In [ ]:
%% 步驟 6：模擬微控制器的控制迴圈
steps = 50;                   % 50 步 x 0.1 秒 = 5 秒
x = [0; 0];                   % 初始狀態：角度 0 度、角速度 0 度/秒
r = 30;                       % 目標指令：轉到 30 度

y_history = zeros(1, steps);  % 預先配置（避免迴圈中反覆擴張陣列）
u_history = zeros(1, steps);
x_history = zeros(2, steps);

for k = 1:steps
    u = -K * x + N * r;       % (1) 控制器：算出這一步該出多少轉矩
    x_next = A * x + B * u;   % (2) 受控體：物理世界演化 0.1 秒
    y = C * x;                % (3) 感測器：讀出「當下」的角度

    y_history(k) = y;         % (4) 記錄
    u_history(k) = u;
    x_history(:,k) = x;
    x = x_next;               % (5) 時間推進
end

% ---- 把狀態方程式逐格攤開，看誰不變、誰在變 ----
disp('===== 常數：設計階段決定，每一步都一樣（都沒有括號）=====');
printf('  A = [%-6g %-6g]    B = [%-8.4f]    C = [%g  %g]    D = %g\n', A(1,1),A(1,2), B(1), C(1),C(2), D);
printf('      [%-6g %-6g]        [%-8.4f]\n', A(2,1),A(2,2), B(2));
printf('  K = [%g  %g]     N = %g     T = %g s     r = %g 度\n\n', K(1),K(2), N, T, r);

disp('===== 每一步都變：帶 (k) 的都在這裡 =====');
printf('%-3s %-7s %-20s %-10s %-20s %-20s %s\n', ...
       'k','t=k*T','x(k)=[角度;角速度]','u(k)','A*x(k)','B*u(k)','x(k+1)=A*x+B*u');
for i = 1:5
    xi = x_history(:,i); ui = u_history(i);
    Ax = A*xi; Bu = B*ui; xn = Ax + Bu;
    printf('%-3d %-7.1f [%7.4f;%8.4f] %10.4f [%7.4f;%8.4f] [%7.4f;%8.4f] [%7.4f;%8.4f]\n', ...
           i, (i-1)*T, xi(1),xi(2), ui, Ax(1),Ax(2), Bu(1),Bu(2), xn(1),xn(2));
end
disp('  => 最右欄 = 前兩欄相加，而它會變成下一列的 x(k)');
disp('  => 小寫 k 一直在跳，大寫 K 一動也不動');

printf('\n===== 為什麼 B(2x1) 能乘 u(1x1)？ =====\n');
printf('  維度：B(%dx%d) * u(1x1) -> 內側 %d 與 1 相符，合法；結果 %dx%d，剛好能跟 A*x 相加\n', ...
       rows(B),columns(B), columns(B), rows(B*1), columns(B*1));
printf('  幾何：B 是「一單位輸入造成的狀態變化方向」，u 是「這一步走多遠」\n');
printf('    B       = [%.4f; %.4f]   <- u = 1 N-m 持續 %g 秒的效果\n', B(1),B(2), T);
printf('    B * 120 = [%.4f; %.4f]   <- u = 120 就是同方向拉長 120 倍\n', B(1)*120, B(2)*120);
printf('    對照上表第 1 列：x(1)=0 所以 A*x(1)=0，於是 x(2) = B*120，完全吻合\n');
disp(' ');

printf('第 1 步的控制量 u(1) = %.1f N-m\n', u_history(1));
printf('第 %d 步（%.1f 秒）的角度 = %.4f 度\n', steps, steps*T, y_history(end));
overshoot = max(0, (max(y_history) - r) / r * 100);   % 超越量只算「超過目標」的部分
printf('整段模擬的最大角度      = %.4f 度  (超越量 = %.2f%%，兩個實極點 → 無震盪)\n', ...
       max(y_history), overshoot);

### 逐行拆解與維度檢查

| 行 | 維度 | 說明 |
|---|---|---|
| `-K * x` | (1×2)(2×1) = **1×1 純量** | 兩個狀態的加權總和 |
| `N * r` | 純量 × 純量 | 目標指令的前饋補償 |
| `A * x` | (2×2)(2×1) = 2×1 | 系統自己的慣性演化 |
| `B * u` | (2×1)(1×1) = 2×1 | 這一步輸入造成的改變 |
| `C * x` | (1×2)(2×1) = 1×1 | 只取出角度 $x_1$ |

> **💡 順序細節**：程式**先記錄再更新**，所以 `y_history(1)` 是初始角度 0，`y_history(k)` 對應時間 $t=(k-1)T$。這種 off-by-one 在數位控制中很常見，畫圖對時間軸時要留意。

### 第一步的控制量：一個實務警訊

$k=1$ 時 $\mathbf{x}=[0,0]^T$，所以 $u(1)=N\times r=4\times30=120$ N·m。

**120 N·m 是非常大的轉矩**，真實推力器不可能輸出，一定會**飽和**（saturation）——而飽和是非線性現象，整套線性設計理論在飽和期間全部失效（這正是 CHP1 第五節提到的放大器飽和議題）。

實務處理方式：(1) 極點選得更靠近 1（慢一點、力道小一點）；(2) 不用階躍指令，改用**斜坡或 S 曲線**讓 $r$ 慢慢升上去；(3) 程式中加入飽和限制與抗飽和機制。**這是教科書例題與真實硬體之間最大的落差之一。**

---


In [ ]:
%% 繪圖：角度響應與控制訊號
t = (0:steps-1) * T;

figure('Position', [100 100 900 700]);

subplot(2,1,1);
stairs(t, y_history, 'b', 'LineWidth', 2); hold on;
plot([t(1) t(end)], [r r], 'r--', 'LineWidth', 1.2);
grid on;
title('衛星姿態響應（極點安置於 z = 0.8, 0.9）');
xlabel('時間 (秒)'); ylabel('角度 (度)');
legend('實際角度 y(k)', '目標 r = 30 度', 'Location', 'southeast');

subplot(2,1,2);
stairs(t, u_history, 'm', 'LineWidth', 2);
grid on;
title('控制訊號 u(k)：推力器轉矩（注意第一步的峰值！）');
xlabel('時間 (秒)'); ylabel('轉矩 (N-m)');

### 結果解讀

**為什麼用 `stairs()` 而不是 `plot()`？** 數位控制的輸入是**階梯狀**的（ZOH 保持 0.1 秒不變），階梯圖才忠實反映「每 0.1 秒才更新一次」的物理實況。用 `plot()` 畫成斜線會誤導你以為訊號是連續變化的。

**上圖（角度）**：平滑爬升、**沒有超越**、沒有震盪。因為兩個極點都是**正實數**（非共軛複數），響應是兩個指數衰減項的疊加，不含振盪成分。若刻意把 $K_2$ 調小，就會看到衝過頭來回震盪——那正是太空中無摩擦系統的本性。

**下圖（控制訊號）**：第一步 120 N·m，之後迅速衰減。這種「開頭爆大力、之後幾乎歸零」的形狀是階躍指令的典型特徵，也是為什麼實務上要用平滑軌跡取代階躍指令。

注意 $u$ 最後趨近於 **0**——呼應第六節的洞察：剛體在太空中維持角度不需要任何轉矩。

---


In [ ]:
%% 補充驗證：5 秒還差 1%，那是穩態誤差還是暫態沒跑完？
long_steps = 200;                    % 20 秒
x = [0; 0]; yl = zeros(1, long_steps);
for k = 1:long_steps
    u = -K * x + N * r;
    yl(k) = C * x;          % 先記錄，再更新 —— 與上面主迴圈的順序一致
    x = A * x + B * u;
end

printf('  50 步 ( 5 秒)：y = %.4f 度   誤差 %.2f%%\n', yl(50),  (r-yl(50))/r*100);
printf(' 100 步 (10 秒)：y = %.4f 度   誤差 %.2f%%\n', yl(100), (r-yl(100))/r*100);
printf(' 200 步 (20 秒)：y = %.6f 度   誤差 %.4f%%\n', yl(200), (r-yl(200))/r*100);

### 解讀

5 秒時的 1% 差距**不是穩態誤差，是暫態還沒衰減完**。主極點 $z=0.9$ 的時間常數 0.949 秒，$4\tau\approx3.8$ 秒才到 2%，5 秒時剩約 1% 完全合理。跑到 20 秒已經是 30.000000——**真正的穩態誤差為零，證明 $N$ 的補償正確**。

判斷「穩態誤差 vs. 暫態未收斂」是新手最常混淆的一點：**把模擬時間拉長 3～5 倍再看一次**，是最快的判別方法。

---


## 八、公式速查表

| 步驟 | 公式 | 用途 | 程式 |
|---|---|---|---|
| 1 | $J\ddot\theta=v$ | 物理定律 | — |
| 1 | $\dot{\mathbf x}=A_c\mathbf x+B_cu$ | 降階為一階向量式 | `ss(Ac,Bc,Cc,Dc)` |
| 2 | $A=e^{A_cT}$ | 離散化系統矩陣 | `c2d(sys,T,'zoh')` |
| 2 | $B=\left(\int_0^Te^{A_c\tau}d\tau\right)B_c$ | 離散化輸入矩陣 | 同上 |
| 3 | $z=e^{sT}$ | s 平面 ↔ z 平面翻譯 | `log(P)/T` |
| 3 | $\alpha_d(z)=\prod(z-p_i)$ | 期望特徵多項式 | `poly(P)` |
| 3.5 | $\mathcal{C}=[B\ AB\ \cdots]$ | 可控性檢查 | `ctrb(A,B)`, `rank()` |
| 4 | $K=[0\cdots0\ 1]\mathcal{C}^{-1}\alpha_d(A)$ | **阿克曼公式** | `acker(A,B,P)` |
| 4 | $\det(zI-A+BK)=\alpha_d(z)$ | 設計成立的驗證式 | `eig(A-B*K)` |
| 5 | $N=1/[C(I-A+BK)^{-1}B]$ | 消除穩態誤差 | `1/dcgain(sys_cl)` |
| 6 | $u(k)=-K\mathbf x(k)+Nr(k)$ | **控制律（燒進 MCU 的那行）** | `u = -K*x + N*r` |
| 6 | $\mathbf x(k+1)=A\mathbf x(k)+Bu(k)$ | 狀態更新（物理） | `x_next = A*x + B*u` |

## 九、常見錯誤與陷阱

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| 拿**連續**矩陣餵 `acker`，卻用 $z$ 平面極點 | 行為完全不對 | 離散設計全程用 $A,B$ 與 $z$ 極點；連續設計全程用 $A_c,B_c$ 與 $s$ 極點，**絕不混用** |
| 忘了 $N$，寫成 `u = -K*x + r` | 穩態值不等於 30 | 一定要算 $N$ |
| 極點選太靠近 0 | 模擬超快，實機推力器飽和 | 檢查 $u(k)$ 峰值是否在致動器能力內 |
| 系統不可控卻硬用 `acker` | 警告、$K$ 出現天文數字 | 先 `rank(ctrb(A,B))` |
| 高階系統（$n>5$）用 `acker` | 極點沒被安置到指定位置 | 改用 `place()` |
| 假設所有狀態都量得到 | 實機上 $x_2$ 根本沒有感測器 | 需要**狀態觀測器** |
| Octave 忘記 `pkg load control` | `'ss' undefined` | 開頭加上該行 |

## 🎯 本例最大的簡化，以及下一步

程式中 `u = -K * x` 用的是**真實狀態** `x`。但 $C=[1\ 0]$ 代表真實衛星上只有角度計、**沒有角速度計**，$x_2$ 量不到。

實務上必須先設計**狀態觀測器**（observer / Luenberger estimator）估測出 $\hat{x}_2$，再用 $\hat{\mathbf x}$ 取代 $\mathbf x$ 做回授——這稱為「觀測器 + 狀態回授」或補償器設計，而且觀測器增益 $L$ 的求法，正是**把阿克曼公式套用在對偶系統 $(A^T, C^T)$ 上**。也就是說，你今天學會的這條公式，等一下會再用一次。

這是本例之後最自然的下一步。
